In [1]:
# ── Cella 1 · Installazione dipendenze ──────────────────────────────────────
import subprocess, sys

packages = [
    "transformers>=4.40.0",
    "trl>=0.8.6",
    "peft>=0.10.0",
    "bitsandbytes>=0.43.0",
    "accelerate>=0.29.0",
    "datasets>=2.19.0",
    "einops",
]

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + packages
)
print("✅ Dipendenze installate")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.8/760.8 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 96.9 MB/s eta 0:00:00
✅ Dipendenze installate


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


In [2]:
# ── Cella 2 · Setup cartelle (aggira il filesystem read-only di Kaggle) ─────
import os

# /kaggle/working/ è SEMPRE scrivibile su Kaggle
BASE_DIR   = "/kaggle/working"
OUTPUT_DIR = os.path.join(BASE_DIR, "gemma2b-dpo")
CACHE_DIR  = os.path.join(BASE_DIR, "hf_cache")
LOGS_DIR   = os.path.join(BASE_DIR, "logs")

for d in [OUTPUT_DIR, CACHE_DIR, LOGS_DIR]:
    os.makedirs(d, exist_ok=True)

# Reindirizza la cache di HuggingFace nella cartella scrivibile
os.environ["HF_HOME"]              = CACHE_DIR
os.environ["TRANSFORMERS_CACHE"]   = os.path.join(CACHE_DIR, "transformers")
os.environ["HF_DATASETS_CACHE"]    = os.path.join(CACHE_DIR, "datasets")
# Necessario per Gemma su Kaggle (evita errori di tokenizer)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print(f"📁 Output  → {OUTPUT_DIR}")
print(f"📁 Cache   → {CACHE_DIR}")

📁 Output  → /kaggle/working/gemma2b-dpo
📁 Cache   → /kaggle/working/hf_cache


In [3]:
# ── Cella 3 · Verifica GPU ───────────────────────────────────────────────────
import torch

print(f"CUDA disponibile : {torch.cuda.is_available()}")
print(f"Numero di GPU    : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    vram  = props.total_memory / 1024**3
    print(f"  GPU {i}: {props.name} — {vram:.1f} GB VRAM")

CUDA disponibile : True
Numero di GPU    : 2
  GPU 0: Tesla T4 — 14.6 GB VRAM
  GPU 1: Tesla T4 — 14.6 GB VRAM


In [4]:
from datasets import load_from_disk

ds = load_from_disk("/kaggle/input/datasets/lorenzosalis/dataset-dpo-trl")

# Se è un DatasetDict
from datasets import DatasetDict
if isinstance(ds, DatasetDict):
    ds = ds["train"]

ds.to_parquet("/kaggle/working/dataset_dpo.parquet")
print("✅ Salvato in /kaggle/working/dataset_dpo.parquet")

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

✅ Salvato in /kaggle/working/dataset_dpo.parquet


In [5]:
# ⚠️ Modifica DATASET_PATH con il percorso reale del tuo dataset su Kaggle
DATASET_PATH = "/kaggle/working/dataset_dpo.parquet"

# Colonne del dataset
COL_PROMPT   = "prompt"
COL_CHOSEN   = "chosen"
COL_REJECTED = "rejected"

# Passaggio a Gemma 2 2B Instruct
MODEL_ID     = "google/gemma-2-2b-it"

# Iperparametri ottimizzati per evitare OOM su singola T4 (15GB VRAM) dopo riavvio sessione
NUM_EPOCHS        = 2      
BATCH_SIZE        = 1      
GRAD_ACCUM        = 16     
LEARNING_RATE     = 5e-6   
MAX_LENGTH        = 512   # Se riscontri ancora limiti di memoria, puoi ridurlo a 768 o 512
BETA              = 0.1    
WARMUP_RATIO      = 0.1    # Verrà interpretato come 10% di passi di riscaldamento

# LoRA
LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.05

print("✅ Configurazione caricata")

✅ Configurazione caricata


In [6]:
# ── Cella 5 · Caricamento dataset ────────────────────────────────────────────
from datasets import load_dataset, DatasetDict

ext = DATASET_PATH.rsplit(".", 1)[-1].lower()
fmt_map = {"json": "json", "jsonl": "json", "csv": "csv", "parquet": "parquet"}
fmt = fmt_map.get(ext, "json")

raw = load_dataset(fmt, data_files=DATASET_PATH, split="train")

# Rinomina colonne se necessario
rename = {}
for target, src in [("prompt", COL_PROMPT),
                    ("chosen", COL_CHOSEN),
                    ("rejected", COL_REJECTED)]:
    if src != target and src in raw.column_names:
        rename[src] = target
if rename:
    raw = raw.rename_columns(rename)

# Mantieni solo le colonne DPO
raw = raw.select_columns(["prompt", "chosen", "rejected"])

# Split train / eval (90/10) — fondamentale con dataset piccolo
split    = raw.train_test_split(test_size=0.1, seed=42)
ds_train = split["train"]
ds_eval  = split["test"]

print(f"Train: {len(ds_train)} esempi")
print(f"Eval : {len(ds_eval)} esempi")
print("\nEsempio:")
print(ds_train[0])

Generating train split: 0 examples [00:00, ? examples/s]

Train: 236 esempi
Eval : 27 esempi

Esempio:
{'prompt': 'Hereâ€™s a highly challenging, multi-disciplinary coding question that requires synthesis of **low-level optimizations, functional programming, concurrency, and domain-specific knowledge** (e.g., parsing, serialization, and probabilistic modeling). Itâ€™s designed to test deep understanding while being concise (under 2000 tokens).\n\n---\n\n### **Question: "The Parallel Parsing Pipeline with Probabilistic Validation"**\n**Languages:** Rust (for low-level control), Haskell (for functional purity), or Python (with asyncio + multiprocessing).\n**Constraints:**\n- No external libraries (except standard libraries).\n- Must handle edge cases explicitly (e.g., malformed input, race conditions).\n- Optimize for both time and space complexity.\n\n---\n\n#### **Problem Statement**\nYou are building a **high-performance, parallelizable log parser** for a distributed telemetry system. Each log line is a JSON-like string with **variable schem

In [7]:
# ── Cella 5b · Autenticazione HuggingFace ────────────────────────────────────
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secret = UserSecretsClient()
hf_token = secret.get_secret("HF_TOKEN")
login(token=hf_token)
print("✅ Autenticato su HuggingFace")

✅ Autenticato su HuggingFace


In [8]:
# ── Cella 6 · Tokenizer ───────────────────────────────────────────────────────
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    cache_dir=CACHE_DIR,
    trust_remote_code=True,
)

# Gemma non ha pad_token di default
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

tokenizer.padding_side = "right"   # necessario per DPO con causal LM

print(f"Vocab size   : {tokenizer.vocab_size}")
print(f"Pad token    : {tokenizer.pad_token!r}")
print(f"Padding side : {tokenizer.padding_side}")

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Vocab size   : 256000
Pad token    : '<pad>'
Padding side : right


In [9]:
import os
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# Ottimizzazione allocazione memoria PyTorch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Configurazione QLoRA a 4-bit per T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # Abilita l'uso dei Tensor Cores in FP16 su T4
    bnb_4bit_use_double_quant=True,
)

# Caricamento del modello sulla prima T4 (cuda:0)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},                    # Mappa tutto sulla prima GPU T4 ed evita bug multi-GPU
    cache_dir=CACHE_DIR,
    trust_remote_code=True,
    dtype=torch.float16,             # Forza float16 per sfruttare l'hardware della T4
    attn_implementation="eager",          
)

model.config.use_cache = False

from collections import Counter
device_counts = Counter(str(p.device) for p in model.parameters())
print("Distribuzione parametri per device:")
for dev, cnt in device_counts.items():
    print(f"  {dev}: {cnt} tensori")

model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Distribuzione parametri per device:
  cuda:0: 288 tensori


In [10]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepara il modello quantizzato per il training
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

# Configurazione estesa a tutti i moduli lineari per catturare relazioni complesse
lora_config = LoraConfig(
    r=LORA_R,               
    lora_alpha=LORA_ALPHA,      
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj", 
        "gate_proj", "up_proj", "down_proj"
    ], 
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 20,766,720 || all params: 2,635,108,608 || trainable%: 0.7881


In [11]:
import torch
from trl import DPOTrainer, DPOConfig

# Svuota la cache CUDA prima di iniziare
torch.cuda.empty_cache()

dpo_config = DPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_RATIO,
    fp16=False,                             # FONDAMENTALE su T4: attiva i Tensor Cores per velocizzare il calcolo
    bf16=False,                            # Disattivato (rallenterebbe la T4)
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=False,          # Previene i crash di fine training
    save_total_limit=2,
    logging_steps=10,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
    beta=BETA,
    max_length=MAX_LENGTH,
    dataloader_pin_memory=False,
    truncation_mode="keep_start",
    loss_type="sigmoid",
    label_smoothing=0.0,
    disable_dropout=True,
    precompute_ref_log_probs=True,         # Massimizza lo spazio libero sulla VRAM della T4
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_config,
    train_dataset=ds_train,
    eval_dataset=ds_eval,
    processing_class=tokenizer,
)

print("🚀 Avvio training DPO ottimizzato su singola GPU T4...")
train_result = trainer.train()

print("\n📊 Risultati training:")
print(train_result)

Adding EOS to train dataset:   0%|          | 0/236 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/236 [00:00<?, ? examples/s]

[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized 

Adding EOS to eval dataset:   0%|          | 0/27 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/27 [00:00<?, ? examples/s]

...

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


🚀 Avvio training DPO ottimizzato su singola GPU T4...


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
1,No log,0.572914,1.416168,239127.000000,-5.243501,-5.694955,0.482810,0.280315,0.016688,0.714286,0.263626,-419.208069,-335.912189
2,0.625590,0.538401,1.421823,478254.000000,-5.225937,-5.676762,0.483024,0.372098,0.017211,0.714286,0.354887,-418.290233,-335.906958


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]


📊 Risultati training:
TrainOutput(global_step=16, training_loss=0.5880148112773895, metrics={'train_runtime': 1440.3497, 'train_samples_per_second': 0.328, 'train_steps_per_second': 0.011, 'total_flos': 5991482737557504.0, 'train_loss': 0.5880148112773895, 'epoch': 2.0})


In [12]:
# ── Cella 11 · Salvataggio modello finale ─────────────────────────────────────
import os

FINAL_DIR = os.path.join(OUTPUT_DIR, "final")
os.makedirs(FINAL_DIR, exist_ok=True)

# Salva adapter LoRA
trainer.model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

# Salva le metriche di training
metrics = train_result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)
trainer.save_state()

print(f"\n✅ Modello salvato in: {FINAL_DIR}")
print("\nFile salvati:")
for f in os.listdir(FINAL_DIR):
    size = os.path.getsize(os.path.join(FINAL_DIR, f)) / 1024**2
    print(f"  {f:40s} {size:.1f} MB")

***** train metrics *****
  epoch                    =        2.0
  total_flos               =  5580003GF
  train_loss               =      0.588
  train_runtime            = 0:24:00.34
  train_samples_per_second =      0.328
  train_steps_per_second   =      0.011

✅ Modello salvato in: /kaggle/working/gemma2b-dpo/final

File salvati:
  ref                                      0.0 MB
  tokenizer_config.json                    0.0 MB
  chat_template.jinja                      0.0 MB
  adapter_config.json                      0.0 MB
  tokenizer.json                           32.8 MB
  adapter_model.safetensors                39.7 MB
  README.md                                0.0 MB
